# VISIONX: Vehicle Classifier - Multi-Dataset Training & Overfitting Check

This notebook trains a custom Convolutional Neural Network (CNN) with **Batch Normalization, Dropout, Weight Decay, and Data Augmentation** strictly on the **train folders of both datasets**:
1. `Vehicles.v1i.multiclass/train` (4,311 images)
2. `Vehicles-coco.v2i.multiclass/train` (13,300 images)
**Total Combined Training Data: 17,611 images** across `[Bus, Motorcycle, Car, Truck]`.

## 1. Hardware Device Verification

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Dataset Paths & Configuration

In [ ]:
import os, sys

# Automatic resolution for Local and Colab environments
base_dir = os.path.dirname(os.getcwd()) if os.path.exists(os.path.join(os.path.dirname(os.getcwd()), "Vehicles.v1i.multiclass")) else os.getcwd()
DATASET1_PATH = os.path.join(base_dir, "Vehicles.v1i.multiclass")
DATASET2_PATH = os.path.join(base_dir, "Vehicles-coco.v2i.multiclass")

BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
TARGET_CLASSES = ['Bus', 'Motorcycle', 'Car', 'Truck']

print(f"Dataset 1 path: {DATASET1_PATH} (Exists: {os.path.exists(DATASET1_PATH)})")
print(f"Dataset 2 path: {DATASET2_PATH} (Exists: {os.path.exists(DATASET2_PATH)})")

## 3. Unified Dataset Loader for Both Datasets

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms

class VehicleDataset(Dataset):
    def __init__(self, csv_file, root_dir, dataset_type=1, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        df = pd.read_csv(csv_file)
        df.columns = df.columns.str.strip()
        self.filenames = []
        self.labels = []
        
        for idx in range(len(df)):
            row = df.iloc[idx]
            fname = str(row['filename']).strip()
            if dataset_type == 1:
                bus, moto, car, truck = float(row.get('Bus', 0)), float(row.get('Motorcycle', 0)), float(row.get('car', 0)), float(row.get('truck', 0))
            else:
                bus, moto, car, truck = float(row.get('bus', 0)), float(row.get('motorcycle', 0)), float(row.get('car', 0)), float(row.get('truck', 0))
            self.filenames.append(fname)
            self.labels.append([bus, moto, car, truck])
        self.labels = np.array(self.labels, dtype=np.float32)

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        img_path = os.path.join(self.root_dir, self.filenames[index])
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (128, 128))
        label = torch.tensor(self.labels[index])
        if self.transform:
            image = self.transform(image)
        return image, label

## 4. Custom CNN Architecture (Overfitting-Protected with BatchNorm & Dropout)

In [ ]:
import torch.nn as nn

class CustomCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(CustomCNN, self).__init__()
        # Block 1
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)    # Regularization
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(2, 2)   # 64x64
        
        # Block 2
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)    # Regularization
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(2, 2)   # 32x32
        
        # Block 3
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)    # Regularization
        self.relu3 = nn.ReLU(inplace=True)
        self.pool3 = nn.MaxPool2d(2, 2)   # 16x16
        self.drop_conv = nn.Dropout2d(0.1)
        
        # Classifier Head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),             # Prevents overfitting
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))
        x = self.drop_conv(x)
        x = self.classifier(x)
        return x

## 5. Combine Train Folders of Both Datasets

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Combine only the train folders
ds1_train = VehicleDataset(os.path.join(DATASET1_PATH, 'train', '_classes.csv'), os.path.join(DATASET1_PATH, 'train'), 1, train_transform)
ds2_train = VehicleDataset(os.path.join(DATASET2_PATH, 'train', '_classes.csv'), os.path.join(DATASET2_PATH, 'train'), 2, train_transform)
combined_train = ConcatDataset([ds1_train, ds2_train])
train_loader = DataLoader(combined_train, batch_size=BATCH_SIZE, shuffle=True)

print(f"Combined Train Samples: {len(combined_train)} ({len(ds1_train)} from Dataset 1 + {len(ds2_train)} from Dataset 2)")

## 6. Training with Overfitting Tracking (Train Loss vs Val Loss)

In [ ]:
import torch.optim as optim

model = CustomCNN(num_classes=4).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Validation loader for overfitting check
val_ds1 = VehicleDataset(os.path.join(DATASET1_PATH, 'valid', '_classes.csv'), os.path.join(DATASET1_PATH, 'valid'), 1, eval_transform)
val_ds2 = VehicleDataset(os.path.join(DATASET2_PATH, 'valid', '_classes.csv'), os.path.join(DATASET2_PATH, 'valid'), 2, eval_transform)
val_loader = DataLoader(ConcatDataset([val_ds1, val_ds2]), batch_size=BATCH_SIZE, shuffle=False)

train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    
    ep_train_loss = running_loss / len(combined_train)
    train_losses.append(ep_train_loss)
    
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for v_imgs, v_lbls in val_loader:
            v_imgs, v_lbls = v_imgs.to(DEVICE), v_lbls.to(DEVICE)
            v_loss += criterion(model(v_imgs), v_lbls).item() * v_imgs.size(0)
    ep_val_loss = v_loss / len(val_loader.dataset)
    val_losses.append(ep_val_loss)
    print(f"Epoch [{epoch+1}/{EPOCHS}] -> Train Loss: {ep_train_loss:.4f} | Val Loss: {ep_val_loss:.4f}")

torch.save(model.state_dict(), 'vehicle_classifier.pth')
print("Training Complete! Saved to vehicle_classifier.pth")

## 7. Overfitting Visualization (Loss Curves)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, 'b-o', label='Training Loss')
plt.plot(range(1, EPOCHS + 1), val_losses, 'r--s', label='Validation Loss')
plt.title('Train vs Validation Loss (Overfitting Diagnostic)', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('BCE Loss', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()

## 8. Single Image Prediction (With 'NA' Fallback)

In [ ]:
def classify_image(image_path, threshold=0.50):
    img = Image.open(image_path).convert('RGB')
    inp = eval_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model(inp)
        probs = torch.sigmoid(out).squeeze().cpu().numpy()
    
    scores = {'Bus': float(probs[0]), 'Motorcycle': float(probs[1]), 'Car': float(probs[2]), 'Truck': float(probs[3])}
    best_cls = max(scores, key=scores.get)
    best_val = scores[best_cls]
    
    if best_val < threshold:
        print('Result: 🚫 NA (No recognized vehicle detected)')
    else:
        print(f'Result: {best_cls} ({best_val*100:.1f}% confidence)')
    print('Probabilities:', {k: f'{v*100:.1f}%' for k, v in scores.items()})
    plt.imshow(img)
    plt.axis('off')
    plt.title('Prediction: ' + (best_cls if best_val >= threshold else 'NA'))
    plt.show()